In [3]:
!pip install gradio transformers torch

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import pickle
import os
from transformers import BertTokenizer, BertForSequenceClassification

# --- 1. ĐỊNH NGHĨA MODEL BiLSTM ---
class BiLSTM_Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(BiLSTM_Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # num_layers=2 và dropout=0.3 thường dùng trong các bài tập ABSA
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        # Kết hợp hidden state của 2 chiều
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(hidden)

# --- 2. HÀM LOAD MODELS (TỰ ĐỘNG FIX LỖI THIẾU FILE) ---
def load_models():
    # Load BERT: Thử load local, nếu lỗi (thiếu file bin) thì load online
    try:
        print("Đang thử load BERT local...")
        tokenizer = BertTokenizer.from_pretrained('./')
        model_bert = BertForSequenceClassification.from_pretrained('./')
    except Exception as e:
        print(f"Lỗi load local: {e}. Đang chuyển sang load BERT online để chạy demo...")
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model_bert = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

    model_bert.eval()

    # Load BiLSTM Vocab
    with open('bilstm_absa_vocab.pkl', 'rb') as f:
        vocab = pickle.load(f)

    # Load BiLSTM Weight
    # Lưu ý: vocab_size phải lấy từ len(vocab)
    model_bilstm = BiLSTM_Model(len(vocab), 100, 128, 3)
    try:
        model_bilstm.load_state_dict(torch.load('bilstm_absa_model.pth', map_location='cpu'))
        print("Đã load BiLSTM thành công!")
    except:
        print("Cảnh báo: Không tìm thấy file bilstm_absa_model.pth, mô hình BiLSTM sẽ chạy với trọng số ngẫu nhiên.")

    model_bilstm.eval()

    return tokenizer, model_bert, model_bilstm, vocab

# Thực hiện load một lần duy nhất
tokenizer, model_bert, model_bilstm, vocab = load_models()

# --- 3. HÀM DỰ ĐOÁN ---
def predict(text):
    labels = ["Negative", "Neutral", "Positive"]

    # BERT Prediction
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        out_bert = model_bert(**inputs).logits
        prob_bert = torch.nn.functional.softmax(out_bert, dim=1)[0]
    res_bert = {labels[i]: float(prob_bert[i]) for i in range(3)}

    # BiLSTM Prediction
    # Tokenize đơn giản bằng cách split và tra từ điển vocab
    tokens = [vocab.get(token.lower(), 0) for token in text.split()] # 0 thường là <pad> hoặc <unk>
    if len(tokens) == 0: tokens = [0]
    tokens_tensor = torch.LongTensor([tokens])

    with torch.no_grad():
        out_ltsm = model_bilstm(tokens_tensor)
        prob_ltsm = torch.nn.functional.softmax(out_ltsm, dim=1)[0]
    res_ltsm = {labels[i]: float(prob_ltsm[i]) for i in range(3)}

    return res_bert, res_ltsm

# --- 4. GIAO DIỆN GRADIO ---
with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>🚀 Sentiment Analysis Dashboard</h1>")
    gr.Markdown("So sánh mô hình **BERT (Transformer)** và **BiLSTM (RNN)**.")

    with gr.Row():
        input_t = gr.Textbox(label="Nhập câu đánh giá (Tiếng Anh)", placeholder="Ví dụ: This product is amazing but a bit expensive...", lines=3)

    with gr.Row():
        btn = gr.Button("Phân tích ngay", variant="primary")
        clear = gr.Button("Xóa")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🤖 Kết quả từ BERT")
            out_bert = gr.Label(num_top_classes=3)
        with gr.Column():
            gr.Markdown("### 🧠 Kết quả từ BiLSTM")
            out_ltsm = gr.Label(num_top_classes=3)

    btn.click(predict, inputs=input_t, outputs=[out_bert, out_ltsm])
    clear.click(lambda: (None, None, None), None, [input_t, out_bert, out_ltsm])

# Mở link công khai
demo.launch(share=True, debug=True)

Đang thử load BERT local...
Lỗi load local: Error while deserializing header: incomplete metadata, file not fully covered. Đang chuyển sang load BERT online để chạy demo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_15085/2638885833.py:83

Cảnh báo: Không tìm thấy file bilstm_absa_model.pth, mô hình BiLSTM sẽ chạy với trọng số ngẫu nhiên.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b88eb278aa60e45a39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
